In [179]:
# V7-01 — load linking_v5 output and collapse imaging duplicates *by union*, not by dropping biology

from pathlib import Path
import pandas as pd
import re

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 240)

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

STRUCT_PATH = WORKING / "output_from_linking_v5.csv"

print("V7-01 — ROOT:      ", ROOT)
print("V7-01 — STRUCT_PATH:", STRUCT_PATH)

if not STRUCT_PATH.exists():
    raise FileNotFoundError(f"V7-01: Structural CSV not found at {STRUCT_PATH}")

df_struct_raw = pd.read_csv(STRUCT_PATH)

print("\nV7-01 — df_struct_raw shape:", df_struct_raw.shape)
print("V7-01 — df_struct_raw columns:")
print(list(df_struct_raw.columns))

print("\nV7-01 — basic ROI QC (raw):")
print("  total rows:      ", len(df_struct_raw))
print("  unique roi_dir:  ", df_struct_raw["roi_dir"].nunique())

dups = df_struct_raw[df_struct_raw.duplicated("roi_dir", keep=False)].copy()
print("  duplicated roi_dir rows:", len(dups))

# ─────────────────────────────────────────────
# 1) Collapse by roi_dir with UNION for imaging-sheet fields
#    (no more 'pick first and drop biology')
# ─────────────────────────────────────────────

# Columns we want to take as "first non-null" because they should be the same across duplicates
TAKE_FIRST_COLS = [
    "date_experiment",
    "fish",
    "roi_rel",
    "roi_name",
    "roi_tiffs",
    "roi_dir",
    "dataset",
    "experiment_folder",
    "roi_path_date_yyyymmdd",
    "fish_folder",
    "roi_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "fish_nickname",
    "fish_raw_norm",
    "roi_anatomy_tokens",
    "roi_anatomy",
    "fish_folder_patched",
    "dataset_slug",
    "dataset_slug_norm",
    "sheet_slug_norm",
    "date_mount_yyyymmdd",
    "date_mount",
    "Date imaged",
    "mount_id",
    "plate_date",
    "mount_id_inferred",
    "mount_id_source",
    "plate_key",
    "plate_id_filled",
    "slot_id_filled",
    # NOTE: we will recompute roi_index_within_slot / bruker_roi_id later from plate/slot if needed,
    # so we do *not* treat differences here as fatal.
]

# Imaging-sheet fields where we WANT to union values across rows
UNION_COLS = [
    "ZF female genotype",
    "ZF male genotype",
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
    "Date born",
    "Imaged Locations",
    "Unique Targets with blanks",
    "Unique Targets",
    "Data location",
    "link_source",
]

# Make sure they exist (some may be absent in some versions; skip gracefully)
TAKE_FIRST_COLS = [c for c in TAKE_FIRST_COLS if c in df_struct_raw.columns]
UNION_COLS      = [c for c in UNION_COLS      if c in df_struct_raw.columns]

def _union_series(series: pd.Series) -> str | pd._libs.missing.NAType:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "None", "<NA>")
    ]
    if not vals:
        return pd.NA
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return "|".join(out)

def _agg_roi_group(g: pd.DataFrame) -> pd.Series:
    out = {}
    # 1) first-value columns
    for c in TAKE_FIRST_COLS:
        # just take the first non-null if possible; otherwise plain first
        col = g[c]
        first_non_null = col[col.notna()]
        out[c] = first_non_null.iloc[0] if not first_non_null.empty else col.iloc[0]

    # 2) union columns
    for c in UNION_COLS:
        out[c] = _union_series(g[c])

    return pd.Series(out)

df_struct = (
    df_struct_raw
    .groupby("roi_dir", dropna=False)
    .apply(_agg_roi_group)
    .reset_index(drop=True)
)

print("\nV7-01 — df_struct AFTER union-collapse:")
print("  total rows:", len(df_struct))
print("  unique roi_dir:", df_struct["roi_dir"].nunique())

# Quick sanity check for the mem-organelle case
mem_org = df_struct[df_struct["dataset_slug"] == "20250808_mem_organelle"]
print("\nV7-01 — sample mem-organelle rows AFTER collapse:")
print(
    mem_org[
        [
            "roi_dir",
            "experiment_folder",
            "additional mRNAs injected" if "additional mRNAs injected" in mem_org.columns else "",
            "Unique Targets with blanks" if "Unique Targets with blanks" in mem_org.columns else "",
        ]
    ].head(12)
)

# df_struct is now your canonical structural input for V7-02+

V7-01 — ROOT:       /Users/davekokel/Projects/carp_v2
V7-01 — STRUCT_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/output_from_linking_v5.csv

V7-01 — df_struct_raw shape: (1082, 46)
V7-01 — df_struct_raw columns:
['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset', 'experiment_folder', 'roi_path_date_yyyymmdd', 'fish_folder', 'roi_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'fish_nickname', 'fish_raw_norm', 'roi_anatomy_tokens', 'roi_anatomy', 'fish_folder_patched', 'dataset_slug', 'dataset_slug_norm', 'sheet_slug_norm', 'date_mount_yyyymmdd', 'date_mount', 'Date imaged', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'link_source', 'plate_date', 'mount_id_inferred', 'mount_id_source', 'pla

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_13202/1515074242.py:132: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_roi_group)


In [180]:
# V7-01b — seed treatment_*_names_sheet directly from imaging-sheet columns

import pandas as pd

if "df_struct" not in globals():
    raise NameError("V7-01b: df_struct not found; run V7-01 first.")

df_struct = df_struct.copy()

# ensure columns exist
for col in [
    "additional mRNAs injected",
    "additional plasmids injected",
]:
    if col not in df_struct.columns:
        raise KeyError(f"V7-01b: df_struct is missing imaging column: {col!r}")

# create / overwrite treatment name columns on df_struct
df_struct["treatment_rna_names_sheet"] = (
    df_struct["additional mRNAs injected"].astype("string")
)

df_struct["treatment_plasmid_names_sheet"] = (
    df_struct["additional plasmids injected"].astype("string")
)

print("V7-01b — seeded treatment name columns from imaging:")
print(
    df_struct[
        [
            "roi_dir",
            "experiment_folder",
            "additional mRNAs injected",
            "treatment_rna_names_sheet",
            "additional plasmids injected",
            "treatment_plasmid_names_sheet",
        ]
    ].head(20)
)

# nice sanity peek for mem-organelle, which was previously broken
mem_org = df_struct[df_struct["experiment_folder"] == "20250808_mem_organelle"]
if not mem_org.empty:
    print("\nV7-01b — mem-organelle rows after seeding:")
    print(
        mem_org[
            [
                "roi_dir",
                "experiment_folder",
                "additional mRNAs injected",
                "treatment_rna_names_sheet",
                "Unique Targets with blanks",
            ]
        ].head(20)
    )

# df_enrich should track df_struct from here forward
df_enrich = df_struct.copy()
print("\nV7-01b — df_enrich reset from df_struct; shape:", df_enrich.shape)

V7-01b — seeded treatment name columns from imaging:
                                              roi_dir                          experiment_folder additional mRNAs injected treatment_rna_names_sheet additional plasmids injected treatment_plasmid_names_sheet
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                      <NA>                         <NA>                          <NA>
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                   

In [181]:
# V7-02 — row-wise mapping: treatment_*_names_sheet → RNA / plasmid basecodes

from pathlib import Path
import pandas as pd
import re

if "df_enrich" not in globals():
    raise NameError("V7-02: df_enrich not found; run V7-01b first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"

INJECTED_RNA_PATH     = RAW / "Unique_injected_rna__preview_dqm.xlsx"
INJECTED_PLASMID_PATH = RAW / "Unique_injected_plasmid__preview_dqm.xlsx"

print("V7-02 — INJECTED_RNA_PATH:    ", INJECTED_RNA_PATH)
print("V7-02 — INJECTED_PLASMID_PATH:", INJECTED_PLASMID_PATH)

missing = [p for p in [INJECTED_RNA_PATH, INJECTED_PLASMID_PATH] if not p.exists()]
if missing:
    print("\nV7-02 — MISSING:")
    for p in missing:
        print("  ", p)
    raise FileNotFoundError("V7-02: missing injected_* mapping sheets; see paths above.")

inj_rna     = pd.read_excel(INJECTED_RNA_PATH)
inj_plasmid = pd.read_excel(INJECTED_PLASMID_PATH)

print("\nV7-02 — injected_rna columns:", list(inj_rna.columns))
print("V7-02 — injected_plasmid columns:", list(inj_plasmid.columns))

# ─────────────────────────────────────────────
# 1) Detect columns and build tiny mapping tables
# ─────────────────────────────────────────────

def _norm_name(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    return s or None

# RNA mapping
rna_name_col = None
rna_base_col = None
for c in inj_rna.columns:
    cl = c.strip().lower()
    if "injected_rna" in cl or cl == "rna":
        rna_name_col = c if rna_name_col is None else rna_name_col
    if "plasmid_base_code" in cl or "plasmid base code" in cl:
        rna_base_col = c

if not rna_name_col or not rna_base_col:
    raise KeyError("V7-02: cannot detect injected_rna name/basecode columns.")

rna_map = (
    inj_rna[[rna_name_col, rna_base_col]]
    .rename(columns={rna_name_col: "treatment_name", rna_base_col: "rna_base_code"})
    .dropna(subset=["treatment_name", "rna_base_code"])
)
rna_map["treatment_name_norm"] = rna_map["treatment_name"].apply(_norm_name)
rna_map = rna_map.dropna(subset=["treatment_name_norm"]).drop_duplicates("treatment_name_norm")

print("\nV7-02 — rna_map sample:")
print(rna_map.head(10))

# Plasmid mapping
pl_name_col = None
pl_base_col = None
for c in inj_plasmid.columns:
    cl = c.strip().lower()
    if "injected_plasmid" in cl or "plasmid" in cl:
        pl_name_col = c if pl_name_col is None else pl_name_col
    if "plasmid_base_code" in cl or "plasmid base code" in cl:
        pl_base_col = c

if pl_name_col and pl_base_col:
    plasmid_map = (
        inj_plasmid[[pl_name_col, pl_base_col]]
        .rename(columns={pl_name_col: "treatment_name", pl_base_col: "plasmid_base_code"})
        .dropna(subset=["treatment_name", "plasmid_base_code"])
    )
    plasmid_map["treatment_name_norm"] = plasmid_map["treatment_name"].apply(_norm_name)
    plasmid_map = plasmid_map.dropna(subset=["treatment_name_norm"]).drop_duplicates("treatment_name_norm")
else:
    plasmid_map = pd.DataFrame(columns=["treatment_name_norm", "plasmid_base_code"])

print("\nV7-02 — plasmid_map sample:")
print(plasmid_map.head(10))

# Build a quick set of "known basecodes" for fallback (MGCO-01, pDQM117, etc.)
basecodes_known = set(
    list(rna_map["rna_base_code"].dropna().astype(str).unique())
    + list(plasmid_map["plasmid_base_code"].dropna().astype(str).unique())
)

# ─────────────────────────────────────────────
# 2) Apply mapping row-wise on df_enrich
# ─────────────────────────────────────────────

df_enrich = df_enrich.copy()

for col in ["treatment_rna_names_sheet", "treatment_plasmid_names_sheet"]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

df_enrich["treatment_rna_name_norm"]     = df_enrich["treatment_rna_names_sheet"].apply(_norm_name)
df_enrich["treatment_plasmid_name_norm"] = df_enrich["treatment_plasmid_names_sheet"].apply(_norm_name)

# RNA basecodes via map
df_enrich = df_enrich.merge(
    rna_map[["treatment_name_norm", "rna_base_code"]],
    how="left",
    left_on="treatment_rna_name_norm",
    right_on="treatment_name_norm",
)

# Plasmid basecodes via map
df_enrich = df_enrich.merge(
    plasmid_map[["treatment_name_norm", "plasmid_base_code"]],
    how="left",
    left_on="treatment_plasmid_name_norm",
    right_on="treatment_name_norm",
    suffixes=("_rna", "_plasmid_map"),
)

# Fallback: if the treatment name itself is a known basecode, use it directly
df_enrich["treatment_rna_rna_base_code"] = df_enrich["rna_base_code"].astype("string")

fallback_mask = df_enrich["treatment_rna_rna_base_code"].isna() & df_enrich["treatment_rna_name_norm"].notna()
fallback_names = df_enrich.loc[fallback_mask, "treatment_rna_name_norm"].astype(str)
is_known = fallback_names.isin(basecodes_known)

df_enrich.loc[fallback_mask & is_known, "treatment_rna_rna_base_code"] = fallback_names[is_known].astype("string")

df_enrich["treatment_plasmid_plasmid_base_code"] = df_enrich["plasmid_base_code"].astype("string")

# ─────────────────────────────────────────────
# 3) Sanity checks on key datasets
# ─────────────────────────────────────────────

print("\nV7-02 — sample treatment mapping rows (after row-wise mapping):")
print(
    df_enrich[
        [
            "roi_dir",
            "experiment_folder",
            "treatment_rna_names_sheet",
            "treatment_rna_rna_base_code",
            "treatment_plasmid_names_sheet",
            "treatment_plasmid_plasmid_base_code",
        ]
    ].head(20)
)

for slug in ["20250808_mem_organelle", "20250917_mem-mito", "20250513_skittles"]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV7-02 — sample rows for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "treatment_rna_names_sheet",
                "treatment_rna_rna_base_code",
                "treatment_plasmid_names_sheet",
                "treatment_plasmid_plasmid_base_code",
            ]
        ].head(10)
    )

print("\nV7-02 — done. Next: use genotype_base_codes + treatment_*_base_code for marker rollup (V7-03).")

V7-02 — INJECTED_RNA_PATH:     /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_rna__preview_dqm.xlsx
V7-02 — INJECTED_PLASMID_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/Unique_injected_plasmid__preview_dqm.xlsx

V7-02 — injected_rna columns: ['injected_rna', 'plasmid_base_code']
V7-02 — injected_plasmid columns: ['injected_plasmid', 'plasmid_base_code']

V7-02 — rna_map sample:
                                treatment_name              rna_base_code                          treatment_name_norm
0                          2xCox8A:mScarlet3S2                    MGCO-35                          2xCox8A:mScarlet3S2
1                                  2xCox8A:mSG                    MGCO-01                                  2xCox8A:mSG
2                2xCox8A:mSG,, 2xLynk:mChilada           MGCO-35, pDQM092                2xCox8A:mSG,, 2xLynk:mChilada
3                 2xLynk:mChilada,, 2xLynk:mSG           pDQM092, pDQM047   

In [182]:
# V7-02b — split multi treatment names and map each token → RNA / plasmid basecodes

import re

if "df_enrich" not in globals():
    raise NameError("V7-02b: df_enrich not found; run V7-01b and V7-02 first.")
if "rna_map" not in globals() or "plasmid_map" not in globals():
    raise NameError("V7-02b: rna_map/plasmid_map not found; run V7-02 first.")

def _split_tokens(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    # split on '|' or ',' but keep simple; imaging sheet uses both styles
    parts = re.split(r"[|]", s)
    return [p.strip() for p in parts if p.strip()]

# quick lookup dicts for speed
rna_lookup = {
    k: v
    for k, v in zip(
        rna_map["treatment_name_norm"],
        rna_map["rna_base_code"].astype(str)
    )
}
plasmid_lookup = {
    k: v
    for k, v in zip(
        plasmid_map["treatment_name_norm"],
        plasmid_map["plasmid_base_code"].astype(str)
    )
}

def _map_rna_tokens(name_str):
    tokens = _split_tokens(name_str)
    out = []
    for t in tokens:
        norm = _norm_name(t)
        if not norm:
            continue
        # 1) exact map from injected_rna table
        if norm in rna_lookup:
            out.append(rna_lookup[norm])
            continue
        # 2) fallback: token itself is a known basecode (MGCO-01, pDQM117, etc.)
        if norm in basecodes_known:
            out.append(norm)
            continue
    # dedupe, keep order
    seen = set()
    uniq = []
    for v in out:
        if v not in seen:
            seen.add(v)
            uniq.append(v)
    return "|".join(uniq) if uniq else pd.NA

def _map_plasmid_tokens(name_str):
    tokens = _split_tokens(name_str)
    out = []
    for t in tokens:
        norm = _norm_name(t)
        if not norm:
            continue
        if norm in plasmid_lookup:
            out.append(plasmid_lookup[norm])
            continue
        if norm in basecodes_known:
            out.append(norm)
            continue
    seen = set()
    uniq = []
    for v in out:
        if v not in seen:
            seen.add(v)
            uniq.append(v)
    return "|".join(uniq) if uniq else pd.NA

# apply token-wise mapping, overwriting/augmenting V7-02 results
df_enrich = df_enrich.copy()

df_enrich["treatment_rna_rna_base_code"] = df_enrich.apply(
    lambda row: _map_rna_tokens(row["treatment_rna_names_sheet"])
    if pd.notna(row["treatment_rna_names_sheet"]) and str(row["treatment_rna_names_sheet"]).strip()
    else row.get("treatment_rna_rna_base_code", pd.NA),
    axis=1,
).astype("string")

df_enrich["treatment_plasmid_plasmid_base_code"] = df_enrich.apply(
    lambda row: _map_plasmid_tokens(row["treatment_plasmid_names_sheet"])
    if pd.notna(row["treatment_plasmid_names_sheet"]) and str(row["treatment_plasmid_names_sheet"]).strip()
    else row.get("treatment_plasmid_plasmid_base_code", pd.NA),
    axis=1,
).astype("string")

print("V7-02b — token-wise treatment mapping applied.")

# sanity checks for the tricky datasets
for slug in ["20250808_mem_organelle", "20250917_mem-mito", "20250513_skittles"]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV7-02b — sample rows for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "treatment_rna_names_sheet",
                "treatment_rna_rna_base_code",
                "treatment_plasmid_names_sheet",
                "treatment_plasmid_plasmid_base_code",
            ]
        ].head(10)
    )

V7-02b — token-wise treatment mapping applied.

V7-02b — sample rows for dataset_slug='20250808_mem_organelle':
                                              roi_dir treatment_rna_names_sheet treatment_rna_rna_base_code treatment_plasmid_names_sheet treatment_plasmid_plasmid_base_code
39  /clusterfs/vast/abcabc/Aang_Foundation/2025080...     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
40  /clusterfs/vast/abcabc/Aang_Foundation/2025080...     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
41  /clusterfs/vast/abcabc/Aang_Foundation/2025080...     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
42  /clusterfs/vast/abcabc/Aang_Foundation/2025080...     2xCox8A:mSG|LAMP1:mSG             MGCO-01|MGCO-15                          <NA>                                <NA>
43  /clusterfs/vas

In [183]:
# V7-03 — build constructs_ft: plasmid_code → fluor_code, tag_code, tag_localization

# normalize key columns
constructs = constructs.rename(columns={
    "code": "plasmid_code",
}).copy()

if "plasmid_code" not in constructs.columns:
    raise KeyError("V7-03: constructs missing 'plasmid_code' column.")

# tags: nickname → tag_code, localization → tag_localization
tags_small = tags_cat.rename(columns={
    "nickname": "tag_code",
    "localization": "tag_localization",
})[["tag_code", "tag_localization"]].drop_duplicates()

# main FT table
constructs_ft = (
    constructs[["plasmid_code", "fluor_code", "tag_code"]]
    .drop_duplicates()
    .merge(tags_small, on="tag_code", how="left")
)

required_cols_cf = {"plasmid_code", "fluor_code", "tag_code", "tag_localization"}
missing_cf = required_cols_cf - set(constructs_ft.columns)
if missing_cf:
    raise KeyError(f"V7-03: constructs_ft is missing columns: {sorted(missing_cf)}")

print("V7-03 — constructs_ft sample:")
print(constructs_ft.head(20))

V7-03 — constructs_ft sample:
   plasmid_code fluor_code tag_code tag_localization
0       pDQM001        mSG      NaN              NaN
1       pDQM002        mSG      NaN              NaN
2       pDQM005      tdmSG   2xLynk         membrane
3       pDQM006      tdmSG      NaN              NaN
4       pDQM007      tdmSG      NaN              NaN
5       pDQM008      tdmSG      NaN              NaN
6       pDQM009   mScarlet      NaN              NaN
7       pDQM009     mKate2      NaN              NaN
8       pDQM009   Electra2      NaN              NaN
9       pDQM009       mKOK      NaN              NaN
10      pDQM009      mTFP1      NaN              NaN
11      pDQM010   mScarlet      NaN              NaN
12      pDQM010     mKate2      NaN              NaN
13      pDQM010   Electra2      NaN              NaN
14      pDQM010       mKOK      NaN              NaN
15      pDQM010      mTFP1      NaN              NaN
16      pDQM011   Electra2      NaN              NaN
17      pDQM012 

In [184]:
# V7-03b — parent_hole patch: fill genotype & treatment names from parent_hole_patch_v6.csv

import pandas as pd
import re
from pathlib import Path

if "df_enrich" not in globals():
    raise NameError("V7-03b: df_enrich not found; run V7-01 / V7-01b first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"

PARENT_HOLES_PATH = RAW / "parent_hole_patch_v6.csv"
print("V7-03b — parent_hole_patch path:", PARENT_HOLES_PATH)

if not PARENT_HOLES_PATH.exists():
    raise FileNotFoundError(f"V7-03b: parent_hole_patch_v6.csv not found at {PARENT_HOLES_PATH}")

parent_holes = pd.read_csv(PARENT_HOLES_PATH)

required_cols = [
    "parent_fish_name",
    "plasmid_base_code",
    "allele",
    "injected_rna",
    "injected_plasmid",
]
missing = [c for c in required_cols if c not in parent_holes.columns]
if missing:
    raise KeyError(f"V7-03b: parent_holes missing columns: {missing}")

def _norm_label(s: str | float | None) -> str | None:
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = re.sub(r"^\d{8}[_-]", "", s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"[^a-z0-9]+", "", s)
    s = s.strip()
    return s or None

def _agg_nonempty(series: pd.Series) -> str | None:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "none", "n/a", "na")
    ]
    if not vals:
        return None
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return "|".join(out)

parent_holes = parent_holes.copy()
parent_holes["parent_slug_norm"] = parent_holes["parent_fish_name"].apply(_norm_label)

parent_hole_agg = (
    parent_holes
    .groupby("parent_slug_norm", dropna=True)
    .agg({
        "plasmid_base_code": _agg_nonempty,
        "allele": _agg_nonempty,
        "injected_rna": _agg_nonempty,
        "injected_plasmid": _agg_nonempty,
    })
    .reset_index()
    .rename(columns={
        "plasmid_base_code": "geno_base_codes_v7_holes",
        "allele": "geno_alleles_v7_holes",
        "injected_rna": "inj_rna_names_v7_holes",
        "injected_plasmid": "inj_plasmid_names_v7_holes",
    })
)

print("V7-03b — parent_hole_agg sample:")
print(parent_hole_agg.head(20))

df_enrich = df_enrich.copy()

# ensure slug column
df_enrich["exp_slug_norm"] = df_enrich["experiment_folder"].apply(_norm_label)

# ensure genotype / treatment columns exist before patching
for col in [
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_names_sheet",
    "treatment_plasmid_names_sheet",
]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.NA

# join hole map onto df_enrich
df_enrich = df_enrich.merge(
    parent_hole_agg,
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
    suffixes=("", "_hole"),
)

def _is_empty_str_series(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

# normalize dtypes to string
for col in [
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_names_sheet",
    "treatment_plasmid_names_sheet",
    "geno_base_codes_v7_holes",
    "geno_alleles_v7_holes",
    "inj_rna_names_v7_holes",
    "inj_plasmid_names_v7_holes",
]:
    if col in df_enrich.columns:
        df_enrich[col] = df_enrich[col].astype("string")

# ── patch genotype from hole map where basecodes are missing ──
mask_geno_missing = _is_empty_str_series(df_enrich["genotype_base_codes"])
mask_geno_hole    = df_enrich["geno_base_codes_v7_holes"].notna()

to_patch_geno = mask_geno_missing & mask_geno_hole

df_enrich.loc[to_patch_geno, "genotype_base_codes"]   = df_enrich.loc[to_patch_geno, "geno_base_codes_v7_holes"]
df_enrich.loc[to_patch_geno, "genotype_allele_codes"] = df_enrich.loc[to_patch_geno, "geno_alleles_v7_holes"]

# ── patch treatment RNA / plasmid names from hole map where they are missing ──
mask_rna_missing     = _is_empty_str_series(df_enrich["treatment_rna_names_sheet"])
mask_plasmid_missing = _is_empty_str_series(df_enrich["treatment_plasmid_names_sheet"])

mask_rna_hole     = df_enrich["inj_rna_names_v7_holes"].notna()
mask_plasmid_hole = df_enrich["inj_plasmid_names_v7_holes"].notna()

to_patch_rna     = mask_rna_missing & mask_rna_hole
to_patch_plasmid = mask_plasmid_missing & mask_plasmid_hole

df_enrich.loc[to_patch_rna, "treatment_rna_names_sheet"] = df_enrich.loc[to_patch_rna, "inj_rna_names_v7_holes"]
df_enrich.loc[to_patch_plasmid, "treatment_plasmid_names_sheet"] = df_enrich.loc[to_patch_plasmid, "inj_plasmid_names_v7_holes"]

print("\nV7-03b — genotype patch summary:")
print("  geno_missing before patch:", int(mask_geno_missing.sum()))
print("  geno_hole slugs present:  ", int(mask_geno_hole.sum()))
print("  patched genotype rows:    ", int(to_patch_geno.sum()))

print("\nV7-03b — treatment name patch summary:")
print("  rna_missing before:       ", int(mask_rna_missing.sum()))
print("  rna_hole slugs present:   ", int(mask_rna_hole.sum()))
print("  patched RNA name rows:    ", int(to_patch_rna.sum()))
print("  patched plasmid name rows:", int(to_patch_plasmid.sum()))

print("\nV7-03b — df_enrich shape:", df_enrich.shape)

V7-03b — parent_hole_patch path: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/parent_hole_patch_v6.csv
V7-03b — parent_hole_agg sample:
                       parent_slug_norm geno_base_codes_v7_holes geno_alleles_v7_holes inj_rna_names_v7_holes inj_plasmid_names_v7_holes
0                      ermsgmemmchilada                  pDQM082                   315        MGCO-04,MGCO-01                       None
1                            memhistone          pDQM005,pDQM133               302,324                   None                       None
2                         memkinetocore                  pDQM082                   315                MGCO-57                       None
3                      memmchiladaermsg                  pDQM082                   315        MGCO-01,MGCO-04                       None
4                               memmito                  pDQM082                   315                MGCO-01                       None
5                  

In [185]:
# V7-03c — experiment-level slug→basecode patch via CSV (experiment_hole_patch_v7)

from pathlib import Path
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V7-03c: df_enrich not found; run V7-01..V7-03b first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"

EXP_PATCH_PATH = RAW / "experiment_hole_patch_v7.csv"
print("V7-03c — EXP_PATCH_PATH:", EXP_PATCH_PATH)

if not EXP_PATCH_PATH.exists():
    raise FileNotFoundError(f"V7-03c: experiment hole patch CSV not found at {EXP_PATCH_PATH}")

exp_patch = pd.read_csv(EXP_PATCH_PATH)

# required columns in the patch CSV
required_cols = {
    "dataset_slug",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_base_codes",
    "treatment_plasmid_base_codes",
}
missing = required_cols - set(exp_patch.columns)
if missing:
    raise KeyError(f"V7-03c: experiment_hole_patch_v7.csv missing columns: {sorted(missing)}")

# normalize slug to match df_enrich.dataset_slug
def _norm_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip()

exp_patch = exp_patch.copy()
exp_patch["dataset_slug_norm"] = exp_patch["dataset_slug"].apply(_norm_slug)

# make sure df_enrich has dataset_slug_norm
df_enrich = df_enrich.copy()
df_enrich["dataset_slug_norm"] = df_enrich.get("dataset_slug_norm", df_enrich["dataset_slug"].apply(_norm_slug))

print("V7-03c — exp_patch rows:", len(exp_patch))
print("V7-03c — exp_patch unique dataset_slug_norm:", exp_patch["dataset_slug_norm"].nunique())

# columns in df_enrich we will patch
patch_map = {
    "genotype_base_codes":            "genotype_base_codes",
    "genotype_allele_codes":          "genotype_allele_codes",
    "treatment_rna_base_codes":       "treatment_rna_rna_base_code",
    "treatment_plasmid_base_codes":   "treatment_plasmid_plasmid_base_code",
}

# ensure target cols exist and are string dtype
for target_col in patch_map.values():
    if target_col not in df_enrich.columns:
        df_enrich[target_col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[target_col] = df_enrich[target_col].astype("string")

# do a left merge so patch columns appear alongside df_enrich
merge_cols = ["dataset_slug_norm"] + list(patch_map.keys())
df_enrich = df_enrich.merge(
    exp_patch[merge_cols],
    how="left",
    on="dataset_slug_norm",
    suffixes=("", "_exp_patch"),
)

def _is_empty_str(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

print("V7-03c — patchable columns:", list(patch_map.values()))

# apply patch *only* where df_enrich col is currently empty and patch col has a value
for src_col, tgt_col in patch_map.items():
    patch_col = f"{src_col}_exp_patch"
    if patch_col not in df_enrich.columns:
        continue

    current = df_enrich[tgt_col]
    patch_vals = df_enrich[patch_col].astype("string")

    mask_missing = _is_empty_str(current)
    mask_has_patch = patch_vals.notna() & (patch_vals.astype(str).str.strip() != "")

    to_patch = mask_missing & mask_has_patch

    print(f"\nV7-03c — patch for {tgt_col}:")
    print("  missing before:", int(mask_missing.sum()))
    print("  rows with patch:", int(mask_has_patch.sum()))
    print("  patched:", int(to_patch.sum()))

    df_enrich.loc[to_patch, tgt_col] = patch_vals[to_patch]

# drop the *_exp_patch helper columns
drop_cols = [c for c in df_enrich.columns if c.endswith("_exp_patch")]
df_enrich = df_enrich.drop(columns=drop_cols)

# quick sanity peek for the 8 must-be-full slugs
must_be_full_slugs = [
    "20250805_lifeact_mem-halo",
    "20251028_mem-peroxi",
    "20251028_mem-peroxi2",
    "20250521_skittles_no-membrane",
    "20251017_nuclear_envelope",
    "20251107_mem-kinectocore",
    "20250602_mem",
    "20251017_microtubules",
]

print("\nV7-03c — sample rows after experiment-level patch (must-be-full slugs):")
print(
    df_enrich[
        df_enrich["dataset_slug"].isin(must_be_full_slugs)
    ][[
        "dataset_slug",
        "roi_dir",
        "genotype_base_codes",
        "genotype_allele_codes",
        "treatment_rna_rna_base_code",
        "treatment_plasmid_plasmid_base_code",
    ]].head(40)
)

print("\nV7-03c — done. Now re-run V7-07b (marker rollup) and V7-08c (export).")

V7-03c — EXP_PATCH_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/experiment_hole_patch_v7.csv
V7-03c — exp_patch rows: 9
V7-03c — exp_patch unique dataset_slug_norm: 9
V7-03c — patchable columns: ['genotype_base_codes', 'genotype_allele_codes', 'treatment_rna_rna_base_code', 'treatment_plasmid_plasmid_base_code']

V7-03c — patch for genotype_base_codes:
  missing before: 209
  rows with patch: 15
  patched: 15

V7-03c — patch for genotype_allele_codes:
  missing before: 209
  rows with patch: 15
  patched: 15

V7-03c — sample rows after experiment-level patch (must-be-full slugs):
                      dataset_slug                                            roi_dir genotype_base_codes genotype_allele_codes treatment_rna_rna_base_code treatment_plasmid_plasmid_base_code
27       20250805_lifeact_mem-halo  /clusterfs/vast/abcabc/Aang_Foundation/2025080...                <NA>                  <NA>                        <NA>                                <NA>


In [186]:
# V7-04 — slug normalization + parent_map aggregated genotype

df_enrich = df_struct.copy()

def _norm_label(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = re.sub(r"^\d{8}[_-]", "", s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"[^a-z0-9]+", "", s)
    s = s.strip()
    return s or None

df_enrich["exp_slug_norm"] = df_enrich["experiment_folder"].apply(_norm_label)

parent_map["parent_slug_norm"] = parent_map["parent_fish_name"].apply(_norm_label)

def _agg_nonempty(series: pd.Series) -> str | None:
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "n/a", "na")
    ]
    if not vals:
        return None
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return "|".join(out)

parent_slug_agg = (
    parent_map
    .groupby("parent_slug_norm", dropna=True)
    .agg({
        "plasmid_base_code": _agg_nonempty,
        "allele": _agg_nonempty,
        "injected_rna": _agg_nonempty,
        "injected_plasmid": _agg_nonempty,
    })
    .reset_index()
    .rename(columns={
        "plasmid_base_code": "geno_base_codes_v7_map",
        "allele": "geno_alleles_v7_map",
        "injected_rna": "inj_rna_names_v7_map",
        "injected_plasmid": "inj_plasmid_names_v7_map",
    })
)

print("V7-04 — parent_slug_agg sample:")
print(parent_slug_agg.head(20))

df_enrich = df_enrich.merge(
    parent_slug_agg,
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
)

# initialize genotype/treatment columns as strings
for col in ["genotype_base_codes", "genotype_allele_codes", "treatment_rna_names_sheet", "treatment_plasmid_names_sheet"]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([np.nan] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

print("\nV7-04 — df_enrich after parent_map join:")
print("  shape:", df_enrich.shape)
print("  columns:", [c for c in df_enrich.columns if c.startswith("geno_") or c.startswith("inj_")][:10])

V7-04 — parent_slug_agg sample:
                               parent_slug_norm geno_base_codes_v7_map  geno_alleles_v7_map inj_rna_names_v7_map inj_plasmid_names_v7_map
0                                           abe                pDQM034                  309                 None                     None
1                                           ben                pDQM034                  310                 None                     None
2                                     casperrnf                   None                 None                 None                     None
3                                         chris                pDQM034                  317                 None                     None
4                                  csppiglet14a                   None                 None                 None                     None
5                                        dennis                pDQM036                  318                 None                     None
6 

In [187]:
# V7-05 — dataset-level hole patch (parent_holes) → genotype + treatment names

parent_holes["parent_slug_norm"] = parent_holes["parent_fish_name"].apply(_norm_label)

parent_hole_agg = (
    parent_holes
    .groupby("parent_slug_norm", dropna=True)
    .agg({
        "plasmid_base_code": _agg_nonempty,
        "allele": _agg_nonempty,
        "injected_rna": _agg_nonempty,
        "injected_plasmid": _agg_nonempty,
    })
    .reset_index()
    .rename(columns={
        "plasmid_base_code": "geno_base_codes_v7_holes",
        "allele": "geno_alleles_v7_holes",
        "injected_rna": "inj_rna_names_v7_holes",
        "injected_plasmid": "inj_plasmid_names_v7_holes",
    })
)

print("V7-05 — parent_hole_agg sample:")
print(parent_hole_agg.head(20))

df_enrich = df_enrich.merge(
    parent_hole_agg,
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
    suffixes=("", "_hole")
)

def _is_empty_str_series(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

# patch genotype from holes where genotype is empty
mask_geno_missing = _is_empty_str_series(df_enrich["genotype_base_codes"])
mask_geno_hole    = df_enrich["geno_base_codes_v7_holes"].notna()

to_patch_geno = mask_geno_missing & mask_geno_hole

df_enrich["geno_base_codes_v7_holes"] = df_enrich["geno_base_codes_v7_holes"].astype("string")
df_enrich["geno_alleles_v7_holes"]    = df_enrich["geno_alleles_v7_holes"].astype("string")

df_enrich.loc[to_patch_geno, "genotype_base_codes"]   = df_enrich.loc[to_patch_geno, "geno_base_codes_v7_holes"]
df_enrich.loc[to_patch_geno, "genotype_allele_codes"] = df_enrich.loc[to_patch_geno, "geno_alleles_v7_holes"]

# patch treatment names from hole map
df_enrich["inj_rna_names_v7_holes"] = df_enrich["inj_rna_names_v7_holes"].astype("string")
df_enrich["inj_plasmid_names_v7_holes"] = df_enrich["inj_plasmid_names_v7_holes"].astype("string")

mask_rna_missing = _is_empty_str_series(df_enrich["treatment_rna_names_sheet"])
mask_plasmid_missing = _is_empty_str_series(df_enrich["treatment_plasmid_names_sheet"])

mask_rna_hole = df_enrich["inj_rna_names_v7_holes"].notna()
mask_plasmid_hole = df_enrich["inj_plasmid_names_v7_holes"].notna()

to_patch_rna = mask_rna_missing & mask_rna_hole
to_patch_plasmid = mask_plasmid_missing & mask_plasmid_hole

df_enrich.loc[to_patch_rna, "treatment_rna_names_sheet"] = df_enrich.loc[to_patch_rna, "inj_rna_names_v7_holes"]
df_enrich.loc[to_patch_plasmid, "treatment_plasmid_names_sheet"] = df_enrich.loc[to_patch_plasmid, "inj_plasmid_names_v7_holes"]

print("\nV7-05 — genotype patch summary:")
print("  geno_missing before patch:", int(mask_geno_missing.sum()))
print("  geno_hole slugs present:  ", int(mask_geno_hole.sum()))
print("  patched genotype rows:    ", int(to_patch_geno.sum()))

print("\nV7-05 — treatment name patch summary:")
print("  rna_missing before:       ", int(mask_rna_missing.sum()))
print("  rna_hole slugs present:   ", int(mask_rna_hole.sum()))
print("  patched RNA name rows:    ", int(to_patch_rna.sum()))
print("  patched plasmid name rows:", int(to_patch_plasmid.sum()))

# hard QA for slugs in parent_holes: genotype + RNA names should not be empty
hole_slugs = sorted(parent_holes["parent_slug_norm"].dropna().unique())
bad_rows = df_enrich[
    df_enrich["exp_slug_norm"].isin(hole_slugs)
    & (
        _is_empty_str_series(df_enrich["genotype_base_codes"])
        | (
            df_enrich["inj_rna_names_v7_holes"].notna()
            & _is_empty_str_series(df_enrich["treatment_rna_names_sheet"])
        )
    )
]

if not bad_rows.empty:
    print("\nV7-05 — ERROR: some hole datasets still lack genotype or RNA names:")
    print(bad_rows[["roi_dir", "experiment_folder", "exp_slug_norm",
                    "genotype_base_codes", "treatment_rna_names_sheet"]].head(40))
    raise ValueError("V7-05: basecode/RNA patch incomplete for some hole datasets.")

print("\nV7-05 — hole slugs look patched. Moving on.")




V7-05 — parent_hole_agg sample:
                       parent_slug_norm geno_base_codes_v7_holes geno_alleles_v7_holes inj_rna_names_v7_holes inj_plasmid_names_v7_holes
0                      ermsgmemmchilada                  pDQM082                   315        MGCO-04,MGCO-01                       None
1                            memhistone          pDQM005,pDQM133               302,324                   None                       None
2                         memkinetocore                  pDQM082                   315                MGCO-57                       None
3                      memmchiladaermsg                  pDQM082                   315        MGCO-01,MGCO-04                       None
4                               memmito                  pDQM082                   315                MGCO-01                       None
5                             memtester                  pDQM082                   315                MGCO-01                       None
6        

In [188]:
# V7-06 — treatment names → basecodes (RNA + plasmid), row-wise, supports multi-names

import re

if "df_enrich" not in globals():
    raise NameError("V7-06: df_enrich not found; run V7-01..V7-05 first.")
if "injected_rna" not in globals() or "injected_plasmid" not in globals():
    raise NameError("V7-06: injected_rna / injected_plasmid not loaded; run V7-02 first.")

df_enrich = df_enrich.copy()

# ─────────────────────────────────────────────
# 1) Small helpers
# ─────────────────────────────────────────────

def _norm_treatment_token(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if not s:
        return None
    return s

def _split_tokens(val):
    """Split a cell that may contain multiple names separated by '|' or ','."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    parts = re.split(r"[|,]", s)
    return [p.strip() for p in parts if p.strip()]

def _uniq(xs):
    out, seen = [], set()
    for x in xs:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# ─────────────────────────────────────────────
# 2) Build lookup dicts from injected_* sheets
# ─────────────────────────────────────────────

# RNA mapping
rna_name_col = None
rna_base_col = None
for c in injected_rna.columns:
    cl = c.strip().lower()
    if "injected_rna" in cl or cl == "rna":
        rna_name_col = c
    if "plasmid_base_code" in cl:
        rna_base_col = c

if not rna_name_col or not rna_base_col:
    raise KeyError("V7-06: cannot detect injected_rna name/basecode columns.")

rna_lookup = {}
for _, row in injected_rna[[rna_name_col, rna_base_col]].iterrows():
    name = _norm_treatment_token(row[rna_name_col])
    bc   = _norm_treatment_token(row[rna_base_col])
    if not name or not bc:
        continue
    # Don't split combos here; treat each row as a whole mapping
    # (combos like '2xLynk:mSG; mChilada:H2B' already appear that way)
    rna_lookup[name] = bc

# Plasmid mapping
pl_name_col = None
pl_base_col = None
for c in injected_plasmid.columns:
    cl = c.strip().lower()
    if "injected_plasmid" in cl or "plasmid" == cl:
        if pl_name_col is None:
            pl_name_col = c
    if "plasmid_base_code" in cl:
        pl_base_col = c

plasmid_lookup = {}
if pl_name_col and pl_base_col:
    for _, row in injected_plasmid[[pl_name_col, pl_base_col]].iterrows():
        name = _norm_treatment_token(row[pl_name_col])
        bc   = _norm_treatment_token(row[pl_base_col])
        if not name or not bc:
            continue
        plasmid_lookup[name] = bc

# Known basecodes from constructs + injected_* (for fallback)
basecodes_known = set()
if "constructs_ft" in globals():
    basecodes_known.update(
        str(x).strip()
        for x in constructs_ft["plasmid_code"].dropna().unique()
    )
for df in (injected_rna, injected_plasmid):
    if "plasmid_base_code" in df.columns:
        basecodes_known.update(
            str(x).strip()
            for x in df["plasmid_base_code"].dropna().unique()
        )

# ─────────────────────────────────────────────
# 3) Row-wise mapping: names → basecode strings
# ─────────────────────────────────────────────

def map_names_to_basecodes(name_string, lookup):
    """
    For a cell like '2xCox8A:mSG|LAMP1:mSG', split into tokens,
    map each to basecode using lookup or fallback if the token is already
    a known basecode (MGCO-xx, pDQMxxx, etc).
    Return 'code1|code2|...' or pd.NA.
    """
    tokens = _split_tokens(name_string)
    codes  = []
    for tok in tokens:
        norm = _norm_treatment_token(tok)
        if not norm:
            continue

        # direct lookup by full token
        if norm in lookup:
            codes.append(lookup[norm])
            continue

        # fallback: token itself is a basecode
        if norm in basecodes_known:
            codes.append(norm)
            continue

    codes = _uniq(codes)
    return "|".join(codes) if codes else pd.NA

# Make sure treatment name columns exist and are string
for col in ["treatment_rna_names_sheet", "treatment_plasmid_names_sheet"]:
    if col not in df_enrich.columns:
        df_enrich[col] = pd.Series([pd.NA] * len(df_enrich), dtype="string")
    else:
        df_enrich[col] = df_enrich[col].astype("string")

df_enrich["treatment_rna_rna_base_code"] = df_enrich["treatment_rna_names_sheet"].apply(
    lambda s: map_names_to_basecodes(s, rna_lookup)
).astype("string")

df_enrich["treatment_plasmid_plasmid_base_code"] = df_enrich["treatment_plasmid_names_sheet"].apply(
    lambda s: map_names_to_basecodes(s, plasmid_lookup)
).astype("string")

# ─────────────────────────────────────────────
# 4) Quick QA: mem-organelle and friends
# ─────────────────────────────────────────────

print("V7-06 — sample treatment mapping rows (after row-wise mapping):")
print(
    df_enrich[
        [
            "roi_dir",
            "experiment_folder",
            "treatment_rna_names_sheet",
            "treatment_rna_rna_base_code",
            "treatment_plasmid_names_sheet",
            "treatment_plasmid_plasmid_base_code",
        ]
    ]
    .head(20)
)

for slug in [
    "20250808_mem_organelle",
    "20250917_mem-mito",
    "20250513_skittles",
]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV7-06 — sample rows for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "treatment_rna_names_sheet",
                "treatment_rna_rna_base_code",
                "treatment_plasmid_names_sheet",
                "treatment_plasmid_plasmid_base_code",
            ]
        ].head(10)
    )

print("\nV7-06 — done. Now re-run V7-07 (marker rollup) and V7-08/V7-09.")

V7-06 — sample treatment mapping rows (after row-wise mapping):
                                              roi_dir                          experiment_folder treatment_rna_names_sheet treatment_rna_rna_base_code treatment_plasmid_names_sheet treatment_plasmid_plasmid_base_code
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                        <NA>                          <NA>                                <NA>
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                        <NA>                          <NA>                                <NA>
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                      <NA>                        <NA>                          <NA>                                <NA>
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hp

In [189]:
# V7-06b — patch constructs_ft for skittles (pDQM034) to be cytosolic

import pandas as pd

if "constructs_ft" not in globals():
    raise NameError("V7-06b: constructs_ft not found; run the constructs_ft build cell first.")

constructs_ft = constructs_ft.copy()

# Make sure tag columns exist
if "tag_code" not in constructs_ft.columns:
    constructs_ft["tag_code"] = pd.NA
if "tag_localization" not in constructs_ft.columns:
    constructs_ft["tag_localization"] = pd.NA

mask_skittles = constructs_ft["plasmid_code"] == "pDQM034"

cols_to_show = [c for c in ["plasmid_code", "fluor_code", "tag_code", "tag_localization"] 
                if c in constructs_ft.columns]

print("V7-06b — pDQM034 rows in constructs_ft BEFORE patch:")
print(constructs_ft.loc[mask_skittles, cols_to_show].head(20))

# For all pDQM034 rows, force a simple "cytosol" tag
constructs_ft.loc[mask_skittles, "tag_code"]         = "cytosol"
constructs_ft.loc[mask_skittles, "tag_localization"] = "cytosol"

print("\nV7-06b — pDQM034 rows in constructs_ft AFTER patch:")
print(constructs_ft.loc[mask_skittles, cols_to_show].head(20))

print("\nV7-06b — done. Now re-run V7-07 (marker rollup) and the export cell.")

V7-06b — pDQM034 rows in constructs_ft BEFORE patch:
   plasmid_code fluor_code tag_code tag_localization
48      pDQM034     mKate2      NaN              NaN
49      pDQM034   mCitrine      NaN              NaN
50      pDQM034   Electra2      NaN              NaN
51      pDQM034       mKOK      NaN              NaN
52      pDQM034      mTFP1      NaN              NaN

V7-06b — pDQM034 rows in constructs_ft AFTER patch:
   plasmid_code fluor_code tag_code tag_localization
48      pDQM034     mKate2  cytosol          cytosol
49      pDQM034   mCitrine  cytosol          cytosol
50      pDQM034   Electra2  cytosol          cytosol
51      pDQM034       mKOK  cytosol          cytosol
52      pDQM034      mTFP1  cytosol          cytosol

V7-06b — done. Now re-run V7-07 (marker rollup) and the export cell.


In [190]:
# V7-06c — manual construct patch for remaining MGCO “hole” datasets

import pandas as pd

# Defensive: make sure constructs_ft exists
if "constructs_ft" not in globals():
    raise NameError("V7-06c: constructs_ft not found; run V7-03 before this cell.")

constructs_ft = constructs_ft.copy()

# Helper to append / upsert simple construct rows
def _add_construct(plasmid_code, fluor_code, tag_code, tag_localization):
    """
    Ensure constructs_ft has at least one row for (plasmid_code, fluor_code)
    with the desired tag_code/tag_localization. If a row already exists,
    we just fill missing tag info.
    """
    mask = (constructs_ft["plasmid_code"] == plasmid_code) & (
        constructs_ft["fluor_code"] == fluor_code
    )

    if mask.any():
        # Fill in missing tag fields
        if "tag_code" in constructs_ft.columns:
            constructs_ft.loc[mask & constructs_ft["tag_code"].isna(), "tag_code"] = tag_code
        if "tag_localization" in constructs_ft.columns:
            constructs_ft.loc[mask & constructs_ft["tag_localization"].isna(), "tag_localization"] = tag_localization
    else:
        # Append a new row with minimal fields
        row = {
            "plasmid_code": plasmid_code,
            "fluor_code": fluor_code,
            "tag_code": tag_code,
            "tag_localization": tag_localization,
        }
        # Make sure all required columns exist
        for col in constructs_ft.columns:
            row.setdefault(col, pd.NA)
        constructs_ft.loc[len(constructs_ft)] = row

# ─────────────────────────────────────────────
# 1) Lifeact constructs (actin)
#    Datasets: 20250805_lifeact_mem-halo, lifeact-mSG, etc.
#    MGCO-24 = Lifeact:Halo  (actin)
#    MGCO-23 = Lifeact:mSG   (actin)
# ─────────────────────────────────────────────
_add_construct("MGCO-24", "Halo", "actin", "actin")
_add_construct("MGCO-23", "mSG",  "actin", "actin")

# ─────────────────────────────────────────────
# 2) Peroxisome constructs
#    Datasets: 20251028_mem-peroxi, 20251028_mem-peroxi2
#    MGCO-49 / MGCO-52 = peroxisome markers
# ─────────────────────────────────────────────
_add_construct("MGCO-49", "mStayGold", "peroxisome", "peroxisome")
_add_construct("MGCO-52", "mStayGold", "peroxisome", "peroxisome")

# ─────────────────────────────────────────────
# 3) Nuclear envelope / kinetochore / microtubule / PCNA
#    Datasets: 20251017_nuclear_envelope, 20251107_mem-kinectocore,
#              20251017_microtubules, 20250801_mem-PCNA
#    These MGCO codes are from your plasmid sheet / imaging sheet.
# ─────────────────────────────────────────────

# MGCO-54 — nuclear envelope (LEMD2 or similar)
_add_construct("MGCO-54", "mStayGold", "nuclear_envelope", "nuclear_envelope")

# MGCO-57 — kinetochore marker
_add_construct("MGCO-57", "mStayGold", "kinetochore", "kinetochore")

# MGCO-50 — microtubules
_add_construct("MGCO-50", "mStayGold", "microtubule", "microtubule")

# tdmChilada:PCNA (in injected_rna as tdmChilada:PCNA / pDQM037)
_add_construct("pDQM037", "tdmChilada", "PCNA", "nucleus")

# ─────────────────────────────────────────────
# 4) Skittles plasmid (pDQM034) — cytosolic multicolor
#    (If you already ran V7-06b you can skip this; otherwise it’s harmless.)
# ─────────────────────────────────────────────
for fl in ["mKate2", "mCitrine", "Electra2", "mKOK", "mTFP1"]:
    _add_construct("pDQM034", fl, "cytosol", "cytosol")

# ─────────────────────────────────────────────
# 5) Mem-only constructs (20250602_mem, etc.)
#    We declare a generic membrane localization for the membrane fluor.
#    If there is no specific organelle, this will at least set 'membrane'.
# ─────────────────────────────────────────────

# MGCO-01 is already mitochondria from your table; for "mem" dataset,
# the membrane fluor is tdmChilada from pDQM082, which already has
# tag_localization = membrane via constructs.
# No extra rows needed here unless you want a specific MGCO-* mem-only.
# (We leave this as-is.)

print("V7-06c — manual construct patch applied.")
print("constructs_ft now has rows for key MGCO hole constructs. "
      "Re-run V7-07b (marker rollup) and V7-08c (export) next.")

V7-06c — manual construct patch applied.
constructs_ft now has rows for key MGCO hole constructs. Re-run V7-07b (marker rollup) and V7-08c (export) next.


In [191]:
# V7-07 — marker rollup and organelles from genotype + treatment basecodes

df_enrich["genotype_base_codes"] = df_enrich["genotype_base_codes"].astype("string")
df_enrich["treatment_rna_rna_base_code"] = df_enrich["treatment_rna_rna_base_code"].astype("string")
df_enrich["treatment_plasmid_plasmid_base_code"] = df_enrich["treatment_plasmid_plasmid_base_code"].astype("string")

def _split_codes(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    parts = [p.strip() for p in re.split(r"[|,]", s) if p.strip()]
    return parts

# genotype markers
geno_rows = []
for _, row in df_enrich[["roi_dir", "genotype_base_codes"]].iterrows():
    codes = _split_codes(row["genotype_base_codes"])
    for code in codes:
        geno_rows.append((row["roi_dir"], code))

geno_df = pd.DataFrame(geno_rows, columns=["roi_dir", "plasmid_code"])
if geno_df.empty:
    geno_markers = pd.DataFrame(columns=[
        "roi_dir",
        "genotype_marker_fluor_codes",
        "genotype_marker_tag_codes",
        "genotype_marker_localizations",
        "genotype_marker_fusion_labels",
    ])
else:
    geno_df = geno_df.merge(constructs_ft, on="plasmid_code", how="left")

    def _agg_geno(group):
        fl = [f for f in group["fluor_code"] if pd.notna(f)]
        tg = [t for t in group["tag_code"] if pd.notna(t)]
        loc = [l for l in group["tag_localization"] if pd.notna(l)]
        def uniq(xs):
            out = []
            seen = set()
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out
        fl_u = uniq(fl)
        tg_u = uniq(tg)
        loc_u = uniq(loc)
        fusion_labels = []
        for f, l in zip(fl, loc):
            if pd.notna(f) and pd.notna(l):
                fusion_labels.append(f"{f}({l})")
        fusion_u = uniq(fusion_labels)
        return pd.Series({
            "genotype_marker_fluor_codes": "|".join(fl_u) if fl_u else pd.NA,
            "genotype_marker_tag_codes": "|".join(tg_u) if tg_u else pd.NA,
            "genotype_marker_localizations": "|".join(loc_u) if loc_u else pd.NA,
            "genotype_marker_fusion_labels": "|".join(fusion_u) if fusion_u else pd.NA,
        })

    geno_markers = (
        geno_df.groupby("roi_dir", dropna=False)
        .apply(_agg_geno)
        .reset_index()
    )

# treatment markers
tx_rows = []
for _, row in df_enrich[["roi_dir", "treatment_rna_rna_base_code", "treatment_plasmid_plasmid_base_code"]].iterrows():
    for code in _split_codes(row["treatment_rna_rna_base_code"]):
        tx_rows.append((row["roi_dir"], code))
    for code in _split_codes(row["treatment_plasmid_plasmid_base_code"]):
        tx_rows.append((row["roi_dir"], code))

tx_df = pd.DataFrame(tx_rows, columns=["roi_dir", "plasmid_code"])
if tx_df.empty:
    tx_markers = pd.DataFrame(columns=[
        "roi_dir",
        "treatment_marker_fluor_codes",
        "treatment_marker_tag_codes",
        "treatment_marker_localizations",
        "treatment_marker_fluor_loc_labels",
    ])
else:
    tx_df = tx_df.merge(constructs_ft, on="plasmid_code", how="left")

    def _agg_tx(group):
        fl = [f for f in group["fluor_code"] if pd.notna(f)]
        tg = [t for t in group["tag_code"] if pd.notna(t)]
        loc = [l for l in group["tag_localization"] if pd.notna(l)]
        def uniq(xs):
            out = []
            seen = set()
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out
        fl_u = uniq(fl)
        tg_u = uniq(tg)
        loc_u = uniq(loc)
        fusion_labels = []
        for f, l in zip(fl, loc):
            if pd.notna(f) and pd.notna(l):
                fusion_labels.append(f"{f}({l})")
        fusion_u = uniq(fusion_labels)
        return pd.Series({
            "treatment_marker_fluor_codes": "|".join(fl_u) if fl_u else pd.NA,
            "treatment_marker_tag_codes": "|".join(tg_u) if tg_u else pd.NA,
            "treatment_marker_localizations": "|".join(loc_u) if loc_u else pd.NA,
            "treatment_marker_fluor_loc_labels": "|".join(fusion_u) if fusion_u else pd.NA,
        })

    tx_markers = (
        tx_df.groupby("roi_dir", dropna=False)
        .apply(_agg_tx)
        .reset_index()
    )

# merge markers into df_enrich
for mdf in [geno_markers, tx_markers]:
    df_enrich = df_enrich.merge(mdf, on="roi_dir", how="left")

# recompute all_unique_organelles / all_fluor_organelles
def _split_pipe(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [p.strip() for p in s.split("|") if p.strip()]

all_unique = []
all_florg = []
for _, row in df_enrich.iterrows():
    locs = _split_pipe(row.get("genotype_marker_localizations", pd.NA)) + \
           _split_pipe(row.get("treatment_marker_localizations", pd.NA))
    fusions = _split_pipe(row.get("genotype_marker_fusion_labels", pd.NA)) + \
              _split_pipe(row.get("treatment_marker_fluor_loc_labels", pd.NA))

    def uniq(xs):
        out = []
        seen = set()
        for x in xs:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    loc_u = uniq(locs)
    fus_u = uniq(fusions)

    all_unique.append("|".join(loc_u) if loc_u else pd.NA)
    all_florg.append("|".join(fus_u) if fus_u else pd.NA)

df_enrich["all_unique_organelles"] = pd.Series(all_unique, dtype="string")
df_enrich["all_fluor_organelles"] = pd.Series(all_florg, dtype="string")

# ─────────────────────────────────────────────
# Final sanity peek (only show columns that actually exist)
# ─────────────────────────────────────────────

cols_show = [
    "roi_dir",
    "genotype_base_codes",
    "treatment_rna_rna_base_code",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
    "all_unique_organelles",
    "all_fluor_organelles",
]

cols_show = [c for c in cols_show if c in df_enrich.columns]

print("V7-07 — marker/organelles sample:")
print(df_enrich[cols_show].head(30))

V7-07 — marker/organelles sample:
                                              roi_dir genotype_base_codes treatment_rna_rna_base_code genotype_marker_fluor_codes genotype_marker_tag_codes genotype_marker_localizations treatment_marker_fluor_codes  \
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                <NA>                        <NA>                         NaN                       NaN                           NaN                          NaN   
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                <NA>                        <NA>                         NaN                       NaN                           NaN                          NaN   
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                <NA>                        <NA>                         NaN                       NaN                           NaN                          NaN   
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                <NA>                     

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_13202/1023647153.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_geno)
/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_13202/1023647153.py:117: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg_tx)


In [192]:
# V7-07b — marker rollup + fallback to imaging-sheet organelles

import re
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V7-07b: df_enrich not found; run earlier V7 cells first.")

# helper to split pipe strings
def _split_pipe(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [p.strip() for p in s.split("|") if p.strip()]

def _is_empty(val):
    if pd.isna(val):
        return True
    return str(val).strip() in ("", "nan", "None", "<NA>")

def _uniq(xs):
    out, seen = [], set()
    for x in xs:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# ---- 1) choose the imaging organelle column ----

sheet_org_col = None
for cand in ["Unique Targets with blanks", "Unique Targets", "unique_targets_sheet"]:
    if cand in df_enrich.columns:
        sheet_org_col = cand
        break

if sheet_org_col is None:
    print("V7-07b WARNING: no imaging organelle column found; fallback will be disabled.")
else:
    print("V7-07b — using imaging organelle column for fallback:", sheet_org_col)

# ---- 2) recompute all_unique_organelles / all_fluor_organelles with fallback ----

all_unique = []
all_fluor  = []

for _, row in df_enrich.iterrows():
    # marker-derived first (preferred)
    locs = _split_pipe(row.get("genotype_marker_localizations", pd.NA)) + \
           _split_pipe(row.get("treatment_marker_localizations", pd.NA))

    flocs = _split_pipe(row.get("genotype_marker_fusion_labels", pd.NA)) + \
            _split_pipe(row.get("treatment_marker_fluor_loc_labels", pd.NA))

    loc_u  = _uniq(locs)
    floc_u = _uniq(flocs)

    # fallback to imaging-sheet organelles ONLY if marker-derived is empty
    if not loc_u and sheet_org_col is not None:
        loc_u = _split_pipe(row.get(sheet_org_col, pd.NA))

    if not floc_u and sheet_org_col is not None:
        # For imaging-only rows, we at least record organelle names as "unknown fluor"
        # e.g. "Mitochondria|Lysosomes" → "Mitochondria(unknown)|Lysosomes(unknown)"
        sheet_locs = _split_pipe(row.get(sheet_org_col, pd.NA))
        floc_u = [f"{loc}(unknown)" for loc in sheet_locs] if sheet_locs else []

    all_unique.append("|".join(loc_u) if loc_u else pd.NA)
    all_fluor.append("|".join(floc_u) if floc_u else pd.NA)

df_enrich["all_unique_organelles"] = pd.Series(all_unique, dtype="string")
df_enrich["all_fluor_organelles"]  = pd.Series(all_fluor, dtype="string")

# ---- 3) quick sanity: show some key datasets ----

print("V7-07b — marker+fallback organelle sample:")
print(
    df_enrich[
        [
            "roi_dir",
            "dataset_slug",
            "genotype_base_codes",
            "treatment_rna_rna_base_code",
            sheet_org_col if sheet_org_col else "dataset_slug",
            "all_unique_organelles",
            "all_fluor_organelles",
        ]
    ].head(30)
)

for slug in [
    "20250808_mem_organelle",
    "20250715_mem_organelle",
    "20250917_mem-mito",
    "20250513_skittles",
]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if not sub.empty:
        print(f"\nV7-07b — sample after fallback for dataset_slug='{slug}':")
        print(
            sub[
                [
                    "roi_dir",
                    "dataset_slug",
                    "genotype_base_codes",
                    "treatment_rna_rna_base_code",
                    sheet_org_col if sheet_org_col else "dataset_slug",
                    "all_unique_organelles",
                    "all_fluor_organelles",
                ]
            ].head(10)
        )

V7-07b — using imaging organelle column for fallback: Unique Targets with blanks
V7-07b — marker+fallback organelle sample:
                                              roi_dir                               dataset_slug genotype_base_codes treatment_rna_rna_base_code Unique Targets with blanks all_unique_organelles                all_fluor_organelles
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                       <NA>                  <NA>                                <NA>
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                       <NA>                  <NA>                                <NA>
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721_72hpf_mrna_mSG_organelle_LLS-SIM                <NA>                        <NA>                       <NA>               

In [193]:
# V7-08c — export from current df_enrich → V7 CSV + DB subset (no recompute)

import pandas as pd
from pathlib import Path

if "df_enrich" not in globals():
    raise NameError("V7-08c: df_enrich not found; run earlier V7 cells first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

FULL_OUT_V7 = WORKING / "legacy_imaging_annotations_v7.csv"
DB_OUT_V7   = WORKING / "legacy_imaging_annotations_for_db_v7.csv"

print("V7-08c — FULL_OUT_V7:", FULL_OUT_V7)
print("V7-08c — DB_OUT_V7:  ", DB_OUT_V7)

# sanity: one row per roi_dir
if df_enrich["roi_dir"].duplicated().any():
    dups = df_enrich[df_enrich["roi_dir"].duplicated(keep=False)]
    print("V7-08c — ERROR: duplicate roi_dir rows in df_enrich; sample:")
    print(dups[["roi_dir", "dataset_slug", "bruker_roi_id"]].head(30))
    raise ValueError("V7-08c: df_enrich has duplicate roi_dir; fix before export.")

print("V7-08c — df_enrich shape:", df_enrich.shape)
print("V7-08c — unique roi_dir:", df_enrich["roi_dir"].nunique())

# ---- 1) full export: just write df_enrich as-is ----
df_enrich.to_csv(FULL_OUT_V7, index=False)

# ---- 2) DB subset: thin view on *current* df_enrich ----
keep_cols = [
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fusion_labels",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

keep_cols = [c for c in keep_cols if c in df_enrich.columns]

df_for_db = df_enrich[keep_cols].copy()

print("\nV7-08c — df_for_db shape:", df_for_db.shape)
print("V7-08c — unique roi_dir in df_for_db:", df_for_db["roi_dir"].nunique())

# quick organelle coverage
missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
)
n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV7-08c — organelle coverage (df_for_db):")
print("  total ROIs:               ", n_total)
print("  ROIs with organelles:     ", n_present)
print("  ROIs missing organelles:  ", n_missing)

print("\nV7-08c — missing organelles by link_source:")
print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))

print("\nV7-08c — top datasets among no-org rows:")
print(
    df_for_db.loc[missing_mask, "dataset_slug"]
    .value_counts()
    .head(20)
)

df_for_db.to_csv(DB_OUT_V7, index=False)

print("\nV7-08c — wrote:")
print("  FULL_OUT_V7:", FULL_OUT_V7)
print("  DB_OUT_V7:  ", DB_OUT_V7)

V7-08c — FULL_OUT_V7: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_v7.csv
V7-08c — DB_OUT_V7:   /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_for_db_v7.csv
V7-08c — df_enrich shape: (976, 71)
V7-08c — unique roi_dir: 976

V7-08c — df_for_db shape: (976, 29)
V7-08c — unique roi_dir in df_for_db: 976

V7-08c — organelle coverage (df_for_db):
  total ROIs:                976
  ROIs with organelles:      935
  ROIs missing organelles:   41

V7-08c — missing organelles by link_source:
link_source
unmatched    22
sheet        19
Name: count, dtype: int64

V7-08c — top datasets among no-org rows:
dataset_slug
Denoising                                    10
20250721_72hpf_mrna_mSG_organelle_LLS-SIM     5
20250805_lifeact_mem-halo                     5
20251028_mem-peroxi                           5
20251028_mem-peroxi2                          5
20250521_skittles_no-membrane              

In [194]:
# V7-09 — QC organelles after final mapping

import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V7-09: df_enrich not found; run V7-01..V7-08 first.")
if "df_for_db" not in globals():
    raise NameError("V7-09: df_for_db not found; run V7-08 first.")

print("V7-09 — df_for_db shape:", df_for_db.shape)

# 1) global coverage
missing_mask = df_for_db["all_unique_organelles"].isna() | (
    df_for_db["all_unique_organelles"].astype(str).str.strip().isin(["", "None", "nan", "<NA>"])
)

n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV7-09 — organelle coverage (df_for_db):")
print("  total ROIs:               ", n_total)
print("  ROIs with organelles:     ", n_present)
print("  ROIs missing organelles:  ", n_missing)

print("\nV7-09 — missing organelles by link_source:")
print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))

print("\nV7-09 — top datasets among no-org rows:")
print(
    df_for_db.loc[missing_mask, "dataset_slug"]
    .value_counts()
    .head(20)
)

# 2) zoom in on the previously-problematic dataset
slug = "20250808_mem_organelle"
sub = df_for_db[df_for_db["dataset_slug"] == slug]

print(f"\nV7-09 — sample rows for dataset_slug='{slug}':")
print(
    sub[
        [
            "roi_dir",
            "dataset_slug",
            "genotype_base_codes",
            "treatment_rna_rna_base_code",
            "all_unique_organelles",
            "all_fluor_organelles",
        ]
    ].head(20)
)

V7-09 — df_for_db shape: (976, 29)

V7-09 — organelle coverage (df_for_db):
  total ROIs:                976
  ROIs with organelles:      935
  ROIs missing organelles:   41

V7-09 — missing organelles by link_source:
link_source
unmatched    22
sheet        19
Name: count, dtype: int64

V7-09 — top datasets among no-org rows:
dataset_slug
Denoising                                    10
20250721_72hpf_mrna_mSG_organelle_LLS-SIM     5
20250805_lifeact_mem-halo                     5
20251028_mem-peroxi                           5
20251028_mem-peroxi2                          5
20250521_skittles_no-membrane                 5
20251017_nuclear_envelope                     2
20251107_mem-kinectocore                      1
20250602_mem                                  1
20251017_microtubules                         1
analysis_test                                 1
Name: count, dtype: int64

V7-09 — sample rows for dataset_slug='20250808_mem_organelle':
                                        

In [195]:
# V7-10 — export remaining “no-organelle” ROIs as a CSV for manual review

from pathlib import Path
import pandas as pd

# ─────────────────────────────────────────────
# 0) Re-establish ROOT/BASE/WORKING
# ─────────────────────────────────────────────
ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

if not WORKING.exists():
    raise FileNotFoundError(f"V7-10: expected WORKING dir at {WORKING}")

# ─────────────────────────────────────────────
# 1) Ensure we have df_for_db
# ─────────────────────────────────────────────
if "df_for_db" not in globals():
    if "df_enrich" not in globals():
        raise NameError("V7-10: df_for_db and df_enrich not found; run earlier V7 cells first.")

    keep_cols = [
        "roi_dir",
        "bruker_roi_id",
        "plate_date",
        "plate_id_filled",
        "slot_id_filled",
        "roi_index_within_slot",
        "dataset_slug",
        "experiment_folder",
        "fish_id",
        "fish_number",
        "fish_age_hpf",
        "roi_anatomy",
        "roi_tiffs",
        "date_experiment",
        "Date imaged",
        "date_mount",
        "genotype_base_codes",
        "genotype_allele_codes",
        "treatment_rna_rna_base_code",
        "treatment_plasmid_plasmid_base_code"
        if "treatment_plasmid_plasmid_base_code" in df_enrich.columns
        else None,
        "all_unique_organelles",
        "all_fluor_organelles",
        "link_source",
    ]
    keep_cols = [c for c in keep_cols if c is not None and c in df_enrich.columns]

    df_for_db = df_enrich[keep_cols].copy()

# sanity: cast organelle cols to string
for col in ["all_unique_organelles", "all_fluor_organelles"]:
    if col in df_for_db.columns:
        df_for_db[col] = df_for_db[col].astype("string")

print("V7-10 — df_for_db shape:", df_for_db.shape)

# ─────────────────────────────────────────────
# 2) Find rows with missing organelles
# ─────────────────────────────────────────────
def _is_empty(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

if "all_unique_organelles" not in df_for_db.columns:
    raise KeyError("V7-10: df_for_db missing all_unique_organelles.")

mask_missing = _is_empty(df_for_db["all_unique_organelles"])
missing = df_for_db.loc[mask_missing].copy()

print("\nV7-10 — missing-org rows:", len(missing))

# ─────────────────────────────────────────────
# 3) Write a compact CSV for manual review
# ─────────────────────────────────────────────
cols_out = [
    "roi_dir",
    "bruker_roi_id",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "roi_anatomy",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_rna_rna_base_code",
    "treatment_plasmid_plasmid_base_code"
    if "treatment_plasmid_plasmid_base_code" in missing.columns
    else None,
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]
cols_out = [c for c in cols_out if c is not None and c in missing.columns]

OUT_PATH = WORKING / "legacy_imaging_missing_organelles_v7.csv"
missing[cols_out].sort_values(["dataset_slug", "roi_dir"]).to_csv(OUT_PATH, index=False)

print("V7-10 — wrote missing-organelle CSV to:")
print(f"  {OUT_PATH}")

print("\nV7-10 — dataset_slug counts among missing-org rows:")
print(missing["dataset_slug"].value_counts())

V7-10 — df_for_db shape: (976, 29)

V7-10 — missing-org rows: 41
V7-10 — wrote missing-organelle CSV to:
  /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_missing_organelles_v7.csv

V7-10 — dataset_slug counts among missing-org rows:
dataset_slug
Denoising                                    10
20250721_72hpf_mrna_mSG_organelle_LLS-SIM     5
20250805_lifeact_mem-halo                     5
20251028_mem-peroxi                           5
20251028_mem-peroxi2                          5
20250521_skittles_no-membrane                 5
20251017_nuclear_envelope                     2
20251107_mem-kinectocore                      1
20250602_mem                                  1
20251017_microtubules                         1
analysis_test                                 1
Name: count, dtype: int64


In [178]:
# V7-11 — hard QA for the 9 “real” hole slugs

import pandas as pd

DB_OUT_V7 = "/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/legacy_imaging_annotations_for_db_v7.csv"

df_final = pd.read_csv(DB_OUT_V7)

must_be_full = [
    "20250721_72hpf_mrna_mSG_organelle_LLS-SIM",
    "20250805_lifeact_mem-halo",
    "20251028_mem-peroxi",
    "20251028_mem-peroxi2",
    "20250521_skittles_no-membrane",
    "20251017_nuclear_envelope",
    "20251107_mem-kinectocore",
    "20250602_mem",
    "20251017_microtubules",
]

mask_missing = (
    df_final["all_unique_organelles"].isna() |
    df_final["all_unique_organelles"].astype(str).str.strip().isin(["", "None", "nan", "<NA>"])
)

bad = df_final[
    df_final["dataset_slug"].isin(must_be_full) & mask_missing
].copy()

print("V7-11 — rows in MUST-BE-FULL slugs that still lack organelles:", len(bad))
if len(bad):
    print(bad[[
        "dataset_slug",
        "roi_dir",
        "genotype_base_codes",
        "genotype_allele_codes",
        "treatment_rna_rna_base_code",
        "treatment_plasmid_plasmid_base_code",
        "all_unique_organelles",
        "all_fluor_organelles",
    ]].head(50))
else:
    print("V7-11 — ✅ all 9 slugs have organelles in the FINAL DB CSV.")

V7-11 — rows in MUST-BE-FULL slugs that still lack organelles: 30
                                  dataset_slug                                            roi_dir genotype_base_codes genotype_allele_codes treatment_rna_rna_base_code treatment_plasmid_plasmid_base_code all_unique_organelles  \
0    20250721_72hpf_mrna_mSG_organelle_LLS-SIM  /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                   NaN                         NaN                                 NaN                   NaN   
1    20250721_72hpf_mrna_mSG_organelle_LLS-SIM  /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                   NaN                         NaN                                 NaN                   NaN   
2    20250721_72hpf_mrna_mSG_organelle_LLS-SIM  /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                   NaN                         NaN                                 NaN                   NaN   
3    20250721_72hp